# Walmart Store Sales — N-BEATS / NBEATSx Training

In [ ]:
!pip install neuralforecast --no-deps -q
!pip install utilsforecast coreforecast --no-deps -q
!pip install wandb -q

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import seaborn as sns
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler
from scipy.stats import loguniform

import wandb
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS, NBEATSx
from neuralforecast.losses.pytorch import MAE
from neuralforecast.tsdataset import TimeSeriesDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

import neuralforecast
print("neuralforecast:", neuralforecast.__version__)
print("torch          :", torch.__version__, "| cuda available:", torch.cuda.is_available())

def _cuda_actually_works():
    if not torch.cuda.is_available():
        return False
    try:
        a = torch.randn(8, device="cuda")
        (a @ a).item()
        return True
    except Exception as e:
        print(f"CUDA reported available but failed ({e}); falling back to CPU.")
        return False

ACCELERATOR = "gpu" if _cuda_actually_works() else "cpu"
print("Using accelerator:", ACCELERATOR)

# Build the dataset

In [ ]:
BASE_PATH = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"
VAL_WEEKS = 8          # held-out weeks for evaluation
HORIZON   = VAL_WEEKS  # N-BEATS forecast horizon
FREQ      = "W-FRI"    # Walmart stamps are always on Fridays
SEASONAL_PERIOD = 52.1775  # 365.25 / 7

raw = pd.read_csv(os.path.join(BASE_PATH, "train.csv.zip"), parse_dates=["Date"])

df = raw.copy()
df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
df = df.rename(columns={"Date": "ds", "Weekly_Sales": "y_raw"})
df = df.sort_values(["unique_id", "ds"]).reset_index(drop=True)
df["y"] = df["y_raw"].astype(float)
df["sample_weight"] = df["IsHoliday"].astype(float).map({1.0: 5.0, 0.0: 1.0})

# ── Fourier terms for annual seasonality (future-known, order=3) ──
def add_fourier_terms(frame, period=SEASONAL_PERIOD, order=3):
    """6 features (sin/cos x 3 harmonics) give a richer seasonality prior than order=2."""
    woy = frame["ds"].dt.isocalendar().week.astype(float).values
    for k in range(1, order + 1):
        frame[f"sin_{k}_woy"] = np.sin(2 * np.pi * k * woy / period)
        frame[f"cos_{k}_woy"] = np.cos(2 * np.pi * k * woy / period)
    return frame

df = add_fourier_terms(df, order=3)
df["IsHoliday_num"] = df["IsHoliday"].astype(float)

FUTR_EXOG = ["IsHoliday_num"] + [c for c in df.columns if c.startswith(("sin_", "cos_"))]
HIST_EXOG = []

features_candidates = ["features.csv.zip", "features.csv"]
features_path = next(
    (p for p in features_candidates if os.path.exists(os.path.join(BASE_PATH, p))), None
)

if features_path is not None:
    feat = pd.read_csv(os.path.join(BASE_PATH, features_path), parse_dates=["Date"])
    feat = feat.rename(columns={"Date": "ds"})
    econ_cols = ["Temperature", "Fuel_Price", "CPI", "Unemployment",
                 "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
    econ_cols = [c for c in econ_cols if c in feat.columns]
    feat = feat[["Store", "ds"] + econ_cols].drop_duplicates(["Store", "ds"])
    feat[econ_cols] = feat[econ_cols].fillna(0.0)
    df = df.merge(feat, on=["Store", "ds"], how="left")
    for c in econ_cols:
        df[c] = df[c].fillna(0.0)
        mu, sigma = df[c].mean(), df[c].std()
        df[f"{c}_z"] = (df[c] - mu) / (sigma if sigma > 1e-8 else 1.0)
    HIST_EXOG = [f"{c}_z" for c in econ_cols]
    print(f"Loaded {features_path}  ->  hist_exog: {HIST_EXOG}")
else:
    print("features.csv not found -- training with calendar-only exogenous features.")

print(f"\nfutr_exog_list ({len(FUTR_EXOG)}): {FUTR_EXOG}")
print(f"hist_exog_list ({len(HIST_EXOG)}): {HIST_EXOG}")

cutoff   = df["ds"].max() - pd.Timedelta(weeks=VAL_WEEKS)
train_df = df[df["ds"] <= cutoff].copy()
val_df   = df[df["ds"] >  cutoff].copy()

series_lengths = train_df.groupby("unique_id").size()
valid_ids = series_lengths[series_lengths > HORIZON].index
train_df  = train_df[train_df["unique_id"].isin(valid_ids)].copy()

print(f"\nDropped {len(series_lengths) - len(valid_ids)} series with <= {HORIZON} weeks of history")
print(f"train: {train_df.shape}   val: {val_df.shape}")
print(f"Unique series for training: {train_df['unique_id'].nunique()}")

NF_COLS = ["unique_id", "ds", "y"] + FUTR_EXOG + HIST_EXOG
nf_train_df = train_df[NF_COLS].reset_index(drop=True)

futr_df = val_df[["unique_id", "ds"] + FUTR_EXOG].reset_index(drop=True)
_nf_trained_ids = set(nf_train_df["unique_id"].unique())
futr_df = futr_df[futr_df["unique_id"].isin(_nf_trained_ids)].reset_index(drop=True)

# Per-series training medians -- used as NaN/Inf fallback in predictions
_series_median = (
    nf_train_df.groupby("unique_id")["y"]
    .median()
    .rename("y_median")
    .reset_index()
)

## Evaluation metric — WMAE

In [ ]:
def wmae(y_true, y_pred, weights):
    return np.average(np.abs(np.asarray(y_true) - np.asarray(y_pred)), weights=weights)


def safe_predict(nf, futr_df_arg, series_median_df):
    """Predict with alignment and NaN/Inf fallback."""
    futr_df_aligned = futr_df_arg
    if futr_df_arg is not None and FUTR_EXOG:
        try:
            expected = nf.make_future_dataframe()
        except Exception:
            try:
                expected = nf.make_future_dataframe(df=nf_train_df)
            except Exception as _e2:
                print(f"  [safe_predict] make_future_dataframe failed ({_e2}); using futr_df as-is.")
                expected = None
        if expected is not None:
            futr_df_aligned = expected.merge(
                futr_df_arg[["unique_id", "ds"] + FUTR_EXOG],
                on=["unique_id", "ds"], how="left",
            )
            for _col in FUTR_EXOG:
                futr_df_aligned[_col] = futr_df_aligned[_col].fillna(0.0)

    try:
        pred_df = (nf.predict(futr_df=futr_df_aligned)
                   if futr_df_aligned is not None else nf.predict())
    except Exception as exc:
        print(f"  [safe_predict] nf.predict() raised {type(exc).__name__}: {exc}. Using median fallback.")
        model_col = nf.models[0].__class__.__name__
        _src = futr_df_aligned if futr_df_aligned is not None else futr_df_arg
        pred_df = _src[["unique_id", "ds"]].copy() if _src is not None else pd.DataFrame()
        pred_df[model_col] = np.nan

    value_cols = [c for c in pred_df.columns if c not in ("unique_id", "ds")]
    had_fallback = False
    for col in value_cols:
        bad_mask = ~np.isfinite(pred_df[col].values)
        if bad_mask.any():
            had_fallback = True
            print(f"  [safe_predict] {bad_mask.sum()} non-finite in '{col}' -- replacing with series median.")
            merged_tmp = pred_df.merge(series_median_df, on="unique_id", how="left")
            pred_df = pred_df.copy()
            pred_df.loc[bad_mask, col] = merged_tmp.loc[bad_mask, "y_median"].values
    return pred_df, had_fallback

## Weights & Biases setup

In [ ]:
wandb.login()  # picks up WANDB_API_KEY from Kaggle Secrets / environment

WANDB_PROJECT = "nbeats-experiment"
print(f"W&B project: {WANDB_PROJECT}")

## Randomised hyperparameter search

In [ ]:
N_SEARCH_ITER       = 25
RANDOM_SEARCH_SEED  = 42

VAL_CHECK_STEPS           = 100
EARLY_STOP_PATIENCE_STEPS = 15

BASELINE_CONFIG = {
    "run_name": "search_00_baseline",
    "params": {
        "architecture":       "interpretable",
        "input_size_mult":    8,
        "n_blocks_per_stack": 3,
        "hidden_width":       256,
        "learning_rate":      3e-4,
        "max_steps":          3000,
        "batch_size":         256,
        "windows_batch_size": 1024,
        "dropout_prob_theta": 0.1,
    },
}

param_distributions = {
    "architecture":        ["generic", "interpretable"],
    # Up to 26*8=208 weeks (approx 4 years) -- sees multi-year seasonality
    "input_size_mult":     [4, 6, 8, 12, 16, 26],
    # 512 is now fine with dropout regularisation
    "hidden_width":        [128, 256, 512],
    "n_blocks_per_stack":  [2, 3, 4],
    # Log-uniform [1e-4, 1e-2]
    "learning_rate":       loguniform(1e-4, 1e-2),
    # More training; early stopping is the safety net
    "max_steps":           [3000, 5000, 7000],
    # Larger batches only -- stable gradients
    "batch_size":          [128, 256],
    # Always >> batch_size
    "windows_batch_size":  [1024, 2048, 4096],
    "dropout_prob_theta":  [0.0, 0.1, 0.2, 0.3],
}

sampled_params = list(ParameterSampler(
    param_distributions, n_iter=N_SEARCH_ITER, random_state=RANDOM_SEARCH_SEED
))

HYPERPARAM_CONFIGS = [BASELINE_CONFIG] + [
    {
        "run_name": f"search_{i+1:02d}",
        "params": {**p, "learning_rate": float(round(p["learning_rate"], 7))},
    }
    for i, p in enumerate(sampled_params)
]

print(f"Sampled {len(HYPERPARAM_CONFIGS)} configurations ({N_SEARCH_ITER} random + 1 baseline)")
for cfg in HYPERPARAM_CONFIGS[:4]:
    print(" ", cfg["run_name"], cfg["params"])
print("  ...")


def build_architecture(architecture, n_blocks_per_stack, hidden_width):
    """Map architecture choice to NBEATSx stack_types / n_blocks / mlp_units."""
    if architecture == "generic":
        stack_types = ["identity", "identity", "identity"]
    else:
        stack_types = ["trend", "seasonality"]
    n_stacks  = len(stack_types)
    n_blocks  = [n_blocks_per_stack] * n_stacks
    mlp_units = [[hidden_width, hidden_width]] * n_stacks
    return stack_types, n_blocks, mlp_units

## Training loop

In [ ]:
results       = []
eval_histories = {}
best_val_wmae  = np.inf
BEST_RUN_NAME, BEST_RUN_ID = None, None
best_merged_df = None

for cfg in HYPERPARAM_CONFIGS:
    run_name = cfg["run_name"]
    p        = cfg["params"]

    stack_types, n_blocks, mlp_units = build_architecture(
        p["architecture"], p["n_blocks_per_stack"], p["hidden_width"]
    )
    input_size = p["input_size_mult"] * HORIZON

    run = wandb.init(
        project=WANDB_PROJECT, name=run_name, group="random_search",
        config={**p, "input_size": input_size, "horizon": HORIZON,
                "val_weeks": VAL_WEEKS, "scaler_type": "robust",
                "val_check_steps": VAL_CHECK_STEPS,
                "early_stop_patience_steps": EARLY_STOP_PATIENCE_STEPS,
                "futr_exog": FUTR_EXOG, "hist_exog": HIST_EXOG},
        reinit=True,
    )

    model = NBEATSx(
        h=HORIZON,
        input_size=input_size,
        futr_exog_list=FUTR_EXOG,
        hist_exog_list=HIST_EXOG,
        stack_types=stack_types,
        n_blocks=n_blocks,
        mlp_units=mlp_units,
        dropout_prob_theta=p["dropout_prob_theta"],
        loss=MAE(),
        max_steps=p["max_steps"],
        learning_rate=p["learning_rate"],
        batch_size=p["batch_size"],
        windows_batch_size=p["windows_batch_size"],
        scaler_type="robust",
        val_check_steps=VAL_CHECK_STEPS,
        early_stop_patience_steps=EARLY_STOP_PATIENCE_STEPS,
        start_padding_enabled=True,
        random_seed=42,
        enable_progress_bar=False,
        logger=False,
        devices=1,
        accelerator=ACCELERATOR,
        gradient_clip_val=1.0,
    )

    nf = NeuralForecast(models=[model], freq=FREQ)
    nf.fit(df=nf_train_df, val_size=HORIZON)

    fitted_model = nf.models[0]
    pred_df, had_fallback = safe_predict(nf, futr_df, _series_median)

    merged = pred_df.merge(
        val_df[["unique_id", "ds", "y_raw", "sample_weight"]],
        on=["unique_id", "ds"], how="inner",
    )
    merged["pred_raw"] = merged["NBEATSx"].values

    val_wmae_val = wmae(merged["y_raw"], merged["pred_raw"], merged["sample_weight"])
    val_mae_val  = float(np.mean(np.abs(merged["y_raw"].values - merged["pred_raw"].values)))
    val_rmse_val = float(np.sqrt(np.mean((merged["y_raw"].values - merged["pred_raw"].values) ** 2)))
    ss_res = np.sum((merged["y_raw"].values - merged["pred_raw"].values) ** 2)
    ss_tot = np.sum((merged["y_raw"].values - merged["y_raw"].mean()) ** 2)
    val_r2_val = float(1 - ss_res / ss_tot) if ss_tot > 0 else float("nan")
    metrics = dict(val_wmae=val_wmae_val, val_mae=val_mae_val,
                   val_rmse=val_rmse_val, val_r2=val_r2_val)

    # ── Overfitting metrics ───────────────────────────────────────────────
    train_traj = fitted_model.train_trajectories
    valid_traj = fitted_model.valid_trajectories

    if valid_traj and len(valid_traj) >= 2:
        val_steps  = [s for s, _ in valid_traj]
        val_losses = [v for _, v in valid_traj]

        min_val_loss   = min(val_losses)
        min_val_idx    = val_losses.index(min_val_loss)
        min_val_step   = val_steps[min_val_idx]
        final_val_loss = val_losses[-1]

        overfit_score = final_val_loss / min_val_loss if min_val_loss > 1e-8 else np.nan
        best_ckpt_fraction = min_val_step / p["max_steps"]

        # Slope of the last 5 validation checkpoints
        tail = list(zip(val_steps, val_losses))[-5:]
        xs_t, ys_t = zip(*tail)
        if len(set(xs_t)) >= 2:
            trend_slope = float(np.polyfit(xs_t, ys_t, 1)[0])
        else:
            trend_slope = np.nan

        final_train_loss_norm = (float(np.mean([v for _, v in train_traj[-5:]]))
                                  if train_traj else np.nan)
    else:
        min_val_loss = final_val_loss = overfit_score = best_ckpt_fraction = trend_slope = np.nan
        final_train_loss_norm = np.nan

    # ── Classify overfitting severity ─────────────────────────────────────
    if pd.isna(overfit_score):
        overfit_level = "unknown"
    elif overfit_score >= 1.5:
        overfit_level = "severe"
    elif overfit_score >= 1.2:
        overfit_level = "moderate"
    else:
        overfit_level = "clean"

    _EMOJI = {"clean": "[OK]", "moderate": "[WARN]", "severe": "[BAD]", "unknown": "[?]"}
    emoji  = _EMOJI[overfit_level]

    wandb.log({
        **metrics,
        "final_train_loss_normalized": final_train_loss_norm,
        "final_val_loss_raw_dollars":  final_val_loss,
        "min_val_loss_raw_dollars":    min_val_loss,
        "overfit_score":               overfit_score,
        "best_ckpt_fraction":          best_ckpt_fraction,
        "valid_trend_slope":           trend_slope,
        "overfit_level":               overfit_level,
        "had_nan_inf_fallback":        had_fallback,
        "n_params":                    sum(t.numel() for t in fitted_model.parameters()),
    })

    results.append({
        "run_name":                    run_name,
        "run_id":                      run.id,
        **p,
        "input_size":                  input_size,
        **metrics,
        "overfit_score":               overfit_score,
        "best_ckpt_fraction":          best_ckpt_fraction,
        "valid_trend_slope":           trend_slope,
        "overfit_level":               overfit_level,
        "final_val_loss_raw_dollars":  final_val_loss,
        "min_val_loss_raw_dollars":    min_val_loss,
        "final_train_loss_normalized": final_train_loss_norm,
        "had_nan_inf_fallback":        had_fallback,
    })
    eval_histories[run_name] = (train_traj, valid_traj)

    if val_wmae_val < best_val_wmae:
        best_val_wmae = val_wmae_val
        BEST_RUN_NAME, BEST_RUN_ID = run_name, run.id
        best_merged_df = merged

    wandb.finish()
    print(f"\n{run_name}: val_wmae={val_wmae_val:.2f}  val_r2={val_r2_val:.4f}  "
          f"({p['architecture']}, input={input_size}, lr={p['learning_rate']:.2e}, "
          f"steps={p['max_steps']}, hidden={p['hidden_width']})")
    print(f"  {emoji} overfit={overfit_level}  overfit_score={overfit_score:.3f}  "
          f"best_ckpt@{best_ckpt_fraction:.1%}  trend_slope={trend_slope:.4f}")

results_df = pd.DataFrame(results)
print(f"\n{'='*70}")
print(f"BEST: {BEST_RUN_NAME}   val_wmae={best_val_wmae:.2f}")
print(f"{'='*70}")

## Results table — colour-coded by overfitting level

In [ ]:
display_cols = [
    "run_name", "architecture", "val_wmae", "val_r2",
    "overfit_score", "best_ckpt_fraction", "overfit_level",
    "learning_rate", "max_steps", "hidden_width", "input_size",
    "dropout_prob_theta",
]


def _color_overfit(val):
    if val == "clean":    return "background-color: #d5f5e3; color: #1e8449"
    if val == "moderate": return "background-color: #fdebd0; color: #784212"
    if val == "severe":   return "background-color: #fadbd8; color: #922b21"
    return ""


def _color_wmae(val):
    if not isinstance(val, (int, float)) or pd.isna(val):
        return ""
    q25 = results_df["val_wmae"].quantile(0.25)
    q75 = results_df["val_wmae"].quantile(0.75)
    if val <= q25: return "background-color: #d5f5e3"
    if val >= q75: return "background-color: #fadbd8"
    return ""


styled = (
    results_df[display_cols]
    .sort_values("val_wmae")
    .reset_index(drop=True)
    .style
    .applymap(_color_overfit, subset=["overfit_level"])
    .applymap(_color_wmae,    subset=["val_wmae"])
    .format({
        "val_wmae":           "{:.2f}",
        "val_r2":             "{:.4f}",
        "overfit_score":      "{:.3f}",
        "best_ckpt_fraction": "{:.1%}",
        "learning_rate":      "{:.2e}",
    })
    .set_caption("All runs sorted by val_wmae | green=clean, orange=moderate, red=severe overfitting")
)
display(styled)

## Overfitting Dashboard

In [ ]:
# Colormap: green (1.0) -> orange -> red (2.0+)
OVERFIT_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "overfit", ["#2ecc71", "#f39c12", "#e74c3c"]
)
SCORE_MIN, SCORE_MAX = 1.0, 2.0


def score_to_color(s):
    if pd.isna(s):
        return "#95a5a6"
    norm_s = (min(max(float(s), SCORE_MIN), SCORE_MAX) - SCORE_MIN) / (SCORE_MAX - SCORE_MIN)
    return OVERFIT_CMAP(norm_s)


fig = plt.figure(figsize=(17, 13))
gs  = fig.add_gridspec(2, 2, hspace=0.40, wspace=0.30)
ax_curves  = fig.add_subplot(gs[0, :])
ax_scatter = fig.add_subplot(gs[1, 0])
ax_hist    = fig.add_subplot(gs[1, 1])

# ── Panel 1: validation loss curves ──────────────────────────────────────
for _, row in results_df.iterrows():
    rname         = row["run_name"]
    _, vtraj      = eval_histories[rname]
    if not vtraj or len(vtraj) < 2:
        continue
    steps, vals = zip(*vtraj)
    color = score_to_color(row["overfit_score"])
    ax_curves.plot(steps, vals, color=color, alpha=0.65, lw=1.4)
    # Mark best checkpoint with a triangle
    min_v = min(vals)
    min_s = steps[list(vals).index(min_v)]
    ax_curves.scatter([min_s], [min_v], color=color, s=45, zorder=5,
                      marker="v", edgecolors="white", linewidths=0.5)

# Highlight the overall best run in black
best_idx_rd  = results_df["val_wmae"].idxmin()
best_rname   = results_df.loc[best_idx_rd, "run_name"]
_, best_vtraj = eval_histories[best_rname]
if best_vtraj:
    bsteps, bvals = zip(*best_vtraj)
    ax_curves.plot(bsteps, bvals, color="black", lw=2.5, zorder=10,
                   label=f"Best run: {best_rname}")
    b_min_v = min(bvals)
    b_min_s = bsteps[list(bvals).index(b_min_v)]
    ax_curves.scatter([b_min_s], [b_min_v], color="black", s=150,
                      zorder=11, marker="*")

ax_curves.set_xlabel("Training step", fontsize=11)
ax_curves.set_ylabel("Internal validation loss (raw $)", fontsize=11)
ax_curves.set_title(
    "Overfitting Dashboard — Validation loss curves over training\n"
    "Colour: green=clean  orange=moderate  red=severe  |  triangle = best checkpoint per run",
    fontsize=12,
)
ax_curves.legend(fontsize=9)
ax_curves.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.1f}k"))

sm = plt.cm.ScalarMappable(
    cmap=OVERFIT_CMAP, norm=mcolors.Normalize(vmin=SCORE_MIN, vmax=SCORE_MAX)
)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax_curves, fraction=0.025, pad=0.01)
cbar.set_label("overfit_score\n(1.0=no degradation, 2.0+=severe)", fontsize=9)

# ── Panel 2: overfit_score vs val_wmae scatter ───────────────────────────
colors_sc = [score_to_color(s) for s in results_df["overfit_score"]]
ax_scatter.scatter(results_df["overfit_score"], results_df["val_wmae"],
                   c=colors_sc, s=75, alpha=0.85,
                   edgecolors="white", linewidths=0.5)
# Star the best run
best_row = results_df.loc[best_idx_rd]
ax_scatter.scatter(best_row["overfit_score"], best_row["val_wmae"],
                   s=220, marker="*", color="black", zorder=10,
                   label="Best (lowest val_wmae)")
ax_scatter.axvline(1.2, color="#f39c12", lw=1.2, ls="--", label="Moderate threshold (1.2)")
ax_scatter.axvline(1.5, color="#e74c3c", lw=1.2, ls="--", label="Severe threshold (1.5)")
ax_scatter.set_xlabel("overfit_score  (final_val / min_val)", fontsize=10)
ax_scatter.set_ylabel("Validation WMAE (lower is better)", fontsize=10)
ax_scatter.set_title("Accuracy vs. Overfitting", fontsize=11)
ax_scatter.legend(fontsize=8)
ax_scatter.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.1f}k"))

# ── Panel 3: best_ckpt_fraction histogram ────────────────────────────────
valid_fracs = results_df["best_ckpt_fraction"].dropna()
ax_hist.hist(valid_fracs, bins=12, color="#3498db", alpha=0.85, edgecolor="white")
med_frac = valid_fracs.median()
ax_hist.axvline(med_frac, color="red", ls="--", lw=1.5,
                label=f"Median: {med_frac:.0%}")
ax_hist.set_xlabel(
    "best_ckpt_fraction\n(fraction of max_steps at which best val was reached)",
    fontsize=9,
)
ax_hist.set_ylabel("# runs", fontsize=10)
ax_hist.set_title("When did each run peak?", fontsize=11)
ax_hist.legend(fontsize=9)
ax_hist.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.suptitle("OVERFITTING DASHBOARD", fontsize=15, fontweight="bold", y=1.01)
plt.savefig("/tmp/plot_overfit_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nOverfitting summary (sorted by val_wmae):")
print(results_df[
    ["run_name", "val_wmae", "overfit_score", "best_ckpt_fraction",
     "valid_trend_slope", "overfit_level"]
].sort_values("val_wmae").to_string(index=False))

## Plot

In [ ]:
order     = results_df.sort_values("val_wmae").reset_index(drop=True)
top_n     = min(20, len(order))
plot_order = order.head(top_n)

# Bar colours by overfitting level
_BAR_COLORS = {"clean": "#2ecc71", "moderate": "#f39c12",
               "severe": "#e74c3c", "unknown": "#95a5a6"}
bar_colors = [_BAR_COLORS.get(ol, "#3498db")
              for ol in plot_order["overfit_level"]]
# Best run always gold
bar_colors[0] = "#f1c40f"

fig, ax = plt.subplots(figsize=(max(10, top_n * 0.6), 5))
bars = ax.bar(plot_order["run_name"], plot_order["val_wmae"], color=bar_colors,
              edgecolor="white", linewidth=0.6)
ax.bar_label(bars, fmt="%.0f", padding=3, fontsize=7)
ax.set_ylabel("Validation WMAE (lower is better)", fontsize=11)
ax.set_title(
    f"Randomised search — val WMAE (best {top_n} of {len(order)} runs)\n"
    "Gold=best | Green=clean fit | Orange=moderate overfit | Red=severe overfit",
    fontsize=11,
)
ax.set_xticklabels(plot_order["run_name"], rotation=45, ha="right", fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.1f}k"))
plt.tight_layout()
plt.savefig("/tmp/plot_wmae_comparison.png", dpi=150)
plt.show()

print("\nTop-5 configs:")
print(order.head(5)[["run_name", "val_wmae", "val_r2", "architecture",
                      "input_size", "learning_rate", "max_steps",
                      "hidden_width", "overfit_level"]].to_string(index=False))

## Plot — learning curves for the top configs


In [ ]:
TOP_N_CURVES = 6
top_runs = results_df.sort_values("val_wmae").head(TOP_N_CURVES)["run_name"].tolist()

ncols = 3
nrows = int(np.ceil(TOP_N_CURVES / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), sharey=False)
axes = np.array(axes).reshape(-1)

for i, rname in enumerate(top_runs):
    train_traj_r, valid_traj_r = eval_histories[rname]
    row_r = results_df[results_df["run_name"] == rname].iloc[0]
    overfit_lvl = row_r["overfit_level"]
    ax = axes[i]

    if valid_traj_r:
        vs, vv = zip(*valid_traj_r)
        ax.plot(vs, vv, lw=2, color="#2c3e50", label="val loss (raw $)")
        # Mark the best checkpoint
        min_v = min(vv)
        min_s = vs[list(vv).index(min_v)]
        ax.axvline(min_s, color="red", lw=1.2, ls="--",
                   label=f"best ckpt @ step {min_s}")
        ax.scatter([min_s], [min_v], s=80, color="red", zorder=5)

    _LCOLOR = {"clean": "#2ecc71", "moderate": "#f39c12",
               "severe": "#e74c3c", "unknown": "gray"}
    overfit_score_r = row_r["overfit_score"]
    score_str = f"{overfit_score_r:.3f}" if pd.notna(overfit_score_r) else "n/a"
    ax.set_title(
        f"{rname}\nWMAE={row_r['val_wmae']:.0f}  "
        f"overfit_score={score_str}  [{overfit_lvl.upper()}]",
        fontsize=9,
        color=_LCOLOR.get(overfit_lvl, "black"),
    )
    ax.set_xlabel("training step", fontsize=8)
    ax.set_ylabel("val loss (raw $)", fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.1f}k"))
    ax.legend(fontsize=7)

for j in range(TOP_N_CURVES, len(axes)):
    axes[j].axis("off")

fig.suptitle(
    f"Learning curves — top {TOP_N_CURVES} configs by val WMAE\n"
    "Red dashed line = best checkpoint (weights restored here by early stopping)",
    y=1.02, fontsize=12,
)
plt.tight_layout()
plt.savefig("/tmp/plot_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Plot — actual vs predicted & residuals (best model, true holdout)

In [ ]:
residuals = best_merged_df["y_raw"].values - best_merged_df["pred_raw"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
ax.scatter(best_merged_df["y_raw"], best_merged_df["pred_raw"],
           s=5, alpha=0.25, color="#3498db")
lims = [
    min(best_merged_df["y_raw"].min(), best_merged_df["pred_raw"].min()),
    max(best_merged_df["y_raw"].max(), best_merged_df["pred_raw"].max()),
]
ax.plot(lims, lims, color="red", lw=1.5, ls="--", label="perfect prediction")
ax.set_xlabel("Actual Weekly Sales", fontsize=11)
ax.set_ylabel("Predicted Weekly Sales", fontsize=11)
ax.set_title(f"Actual vs Predicted — {BEST_RUN_NAME}", fontsize=11)
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

ax = axes[1]
ax.scatter(best_merged_df["pred_raw"], residuals, s=5, alpha=0.25, color="#9b59b6")
ax.axhline(0, color="red", lw=1.5, ls="--")
ax.set_xlabel("Predicted Weekly Sales", fontsize=11)
ax.set_ylabel("Residual (actual - predicted)", fontsize=11)
ax.set_title("Residuals vs Predicted", fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

plt.tight_layout()
plt.savefig("/tmp/plot_actual_vs_pred.png", dpi=150)
plt.show()

print(f"Residual mean : {residuals.mean():.2f}")
print(f"Residual std  : {residuals.std():.2f}")
print(f"Residual p95  : {np.percentile(np.abs(residuals), 95):.2f}")

## Plot — Holiday vs Non-Holiday WMAE breakdown

In [ ]:
hol_mask    = best_merged_df["sample_weight"] > 1.0
non_hol_mask = ~hol_mask

hol_mae   = float(np.mean(np.abs(
    best_merged_df.loc[hol_mask,     "y_raw"] -
    best_merged_df.loc[hol_mask,     "pred_raw"])))
nonhol_mae = float(np.mean(np.abs(
    best_merged_df.loc[non_hol_mask, "y_raw"] -
    best_merged_df.loc[non_hol_mask, "pred_raw"])))
overall_wmae = wmae(
    best_merged_df["y_raw"], best_merged_df["pred_raw"], best_merged_df["sample_weight"]
)

print(f"Holiday MAE     : {hol_mae:>10,.2f}  (n={hol_mask.sum()})")
print(f"Non-Holiday MAE : {nonhol_mae:>10,.2f}  (n={non_hol_mask.sum()})")
print(f"Overall WMAE    : {overall_wmae:>10,.2f}")
print(f"Holiday/Non-Hol ratio: {hol_mae/nonhol_mae:.2f}x  "
      "(< 2.0 is good; > 3.0 means holiday weeks dominate the error)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: MAE bar comparison ──
ax = axes[0]
categories = ["Non-Holiday MAE", "Holiday MAE"]
values     = [nonhol_mae, hol_mae]
colors_hol = ["#3498db", "#e74c3c"]
bars = ax.bar(categories, values, color=colors_hol, edgecolor="white", width=0.5)
ax.bar_label(bars, fmt="${:,.0f}", padding=4, fontsize=10)
ax.set_ylabel("Mean Absolute Error ($)", fontsize=11)
ax.set_title("Holiday vs Non-Holiday MAE", fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax.axhline(overall_wmae, color="#2c3e50", lw=1.5, ls="--",
           label=f"Overall WMAE ${overall_wmae:,.0f}")
ax.legend(fontsize=9)

# ── Right: residuals distributions split ──
ax = axes[1]
hol_resid    = (best_merged_df.loc[hol_mask,     "y_raw"] -
                best_merged_df.loc[hol_mask,     "pred_raw"]).values
nonhol_resid = (best_merged_df.loc[non_hol_mask, "y_raw"] -
                best_merged_df.loc[non_hol_mask, "pred_raw"]).values
ax.hist(nonhol_resid / 1000, bins=60, alpha=0.6, color="#3498db",
        label="Non-Holiday", density=True)
ax.hist(hol_resid    / 1000, bins=30, alpha=0.6, color="#e74c3c",
        label="Holiday",     density=True)
ax.axvline(0, color="black", lw=1.2, ls="--")
ax.set_xlabel("Residual ($000s)", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Residual distributions by holiday flag", fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/plot_holiday_breakdown.png", dpi=150)
plt.show()

## Plot — forecast vs actuals for the top 4 high-volume series

In [ ]:
# Top 4 series by total training sales (high-volume = high WMAE impact)
top4_ids = (
    train_df.groupby("unique_id")["y_raw"]
    .sum()
    .sort_values(ascending=False)
    .head(4)
    .index.tolist()
)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
axes = axes.reshape(-1)

for ax, uid in zip(axes, top4_ids):
    # Training history (last 52 weeks for context)
    hist = train_df[train_df["unique_id"] == uid].tail(52)
    pred = best_merged_df[best_merged_df["unique_id"] == uid]

    ax.plot(hist["ds"], hist["y_raw"] / 1000, color="#95a5a6",
            lw=1.2, alpha=0.7, label="Historical")
    ax.plot(pred["ds"], pred["y_raw"]     / 1000, color="#2c3e50",
            lw=2.0, label="Actual (holdout)")
    ax.plot(pred["ds"], pred["pred_raw"]  / 1000, color="#e74c3c",
            lw=2.0, ls="--", label="Predicted")

    # Shade holiday weeks in the holdout
    for _, row_h in pred.iterrows():
        if row_h["sample_weight"] > 1.0:
            ax.axvspan(row_h["ds"] - pd.Timedelta(days=3),
                       row_h["ds"] + pd.Timedelta(days=3),
                       alpha=0.12, color="gold")

    ser_wmae = wmae(pred["y_raw"], pred["pred_raw"], pred["sample_weight"])
    ax.set_title(f"Store-Dept {uid}  |  Holdout WMAE = ${ser_wmae:,.0f}", fontsize=10)
    ax.set_xlabel("Date")
    ax.set_ylabel("Weekly Sales ($000s)")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=30)

plt.suptitle(
    "Forecast vs Actuals — top 4 high-volume Store-Dept series\n"
    "Gold shading = holiday weeks (weighted 5x in WMAE)",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.savefig("/tmp/plot_top_series_forecast.png", dpi=150, bbox_inches="tight")
plt.show()

## Plot — per-series WMAE distribution

In [ ]:
per_series_wmae = (
    best_merged_df
    .groupby("unique_id")
    .apply(lambda g: wmae(g["y_raw"], g["pred_raw"], g["sample_weight"]))
    .rename("series_wmae")
    .reset_index()
    .sort_values("series_wmae", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: histogram of per-series WMAE ──
ax = axes[0]
ax.hist(per_series_wmae["series_wmae"] / 1000, bins=40,
        color="#3498db", alpha=0.85, edgecolor="white")
ax.axvline(per_series_wmae["series_wmae"].mean()   / 1000,
           color="red",  lw=1.5, ls="--",
           label=f'Mean ${per_series_wmae["series_wmae"].mean():,.0f}')
ax.axvline(per_series_wmae["series_wmae"].median() / 1000,
           color="#2ecc71", lw=1.5, ls="--",
           label=f'Median ${per_series_wmae["series_wmae"].median():,.0f}')
ax.set_xlabel("Per-series WMAE ($000s)", fontsize=11)
ax.set_ylabel("# series", fontsize=11)
ax.set_title("Distribution of per-series WMAE", fontsize=12)
ax.legend(fontsize=9)

# ── Right: worst 15 series by WMAE ──
ax = axes[1]
worst15 = per_series_wmae.head(15)
ax.barh(worst15["unique_id"], worst15["series_wmae"] / 1000,
        color="#e74c3c", alpha=0.85, edgecolor="white")
ax.set_xlabel("WMAE ($000s)", fontsize=11)
ax.set_title("15 hardest-to-forecast series", fontsize=12)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig("/tmp/plot_per_series_error.png", dpi=150)
plt.show()

print(f"Total series in holdout  : {len(per_series_wmae)}")
print(f"Worst single series WMAE : ${per_series_wmae['series_wmae'].max():,.0f}")
top10_pct = per_series_wmae.head(int(len(per_series_wmae) * 0.1))["series_wmae"].sum()
total     = per_series_wmae["series_wmae"].sum()
print(f"Top 10% hardest series account for {top10_pct/total:.1%} of aggregate WMAE")

## Plot — hyperparameter sensitivity

In [ ]:
hp_cols = ["input_size", "learning_rate", "max_steps", "hidden_width",
           "n_blocks_per_stack", "dropout_prob_theta", "windows_batch_size"]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.reshape(-1)

arch_colors = {"generic": "#16a085", "interpretable": "#8e44ad"}

for ax, col in zip(axes, hp_cols):
    for arch, color in arch_colors.items():
        sub = results_df[results_df["architecture"] == arch]
        ax.scatter(sub[col], sub["val_wmae"], s=50, alpha=0.75,
                   color=color, label=arch)
    # Star the best
    ax.scatter(
        results_df.loc[best_idx_rd, col],
        results_df.loc[best_idx_rd, "val_wmae"],
        s=180, color="#f1c40f", marker="*", zorder=5,
        edgecolors="black", linewidths=0.8, label="best",
    )
    if col == "learning_rate":
        ax.set_xscale("log")
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel("val WMAE", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.1f}k"))
    ax.legend(fontsize=7)

for j in range(len(hp_cols), len(axes)):
    axes[j].axis("off")

fig.suptitle("Validation WMAE vs individual hyperparameters\nStar = best config",
             y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig("/tmp/plot_hyperparam_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

## Plot — interpretable decomposition (trend & seasonality)

In [ ]:
DECOMPOSE_INPUT_SIZE_MULT = 8
DECOMPOSE_N_BLOCKS        = 3
DECOMPOSE_HIDDEN          = 128
DECOMPOSE_MAX_STEPS       = 1500

# The decompose() API requires plain NBEATS (not NBEATSx).
# Use a y-only frame (no exog columns).
nf_decompose_df = nf_train_df[["unique_id", "ds", "y"]].copy()

decompose_model = NBEATS(
    h=HORIZON,
    input_size=DECOMPOSE_INPUT_SIZE_MULT * HORIZON,
    stack_types=["trend", "seasonality"],
    n_blocks=[DECOMPOSE_N_BLOCKS, DECOMPOSE_N_BLOCKS],
    mlp_units=[[DECOMPOSE_HIDDEN, DECOMPOSE_HIDDEN]] * 2,
    n_harmonics=3,
    n_polynomials=3,
    loss=MAE(),
    max_steps=DECOMPOSE_MAX_STEPS,
    scaler_type="robust",
    random_seed=42,
    start_padding_enabled=True,
    enable_progress_bar=False,
    logger=False,
    devices=1,
    accelerator=ACCELERATOR,
    gradient_clip_val=1.0,
)
nf_decompose = NeuralForecast(models=[decompose_model], freq=FREQ)
nf_decompose.fit(df=nf_decompose_df)

example_ids = (
    train_df.groupby("unique_id")["y_raw"]
    .sum()
    .sort_values(ascending=False)
    .head(2)
    .index.tolist()
)

fig, axes = plt.subplots(len(example_ids), 1, figsize=(12, 5 * len(example_ids)))
axes = np.atleast_1d(axes)

for ax, uid in zip(axes, example_ids):
    try:
        series_df = nf_decompose_df[nf_decompose_df["unique_id"] == uid]
        ts_dataset, *_ = TimeSeriesDataset.from_df(
            series_df, id_col="unique_id", time_col="ds", target_col="y"
        )
        blocks = nf_decompose.models[0].decompose(ts_dataset)[0]

        level       = blocks[0]
        trend       = blocks[1:1 + DECOMPOSE_N_BLOCKS].sum(axis=0)
        seasonality = blocks[1 + DECOMPOSE_N_BLOCKS:].sum(axis=0)
        total       = blocks.sum(axis=0)

        steps = np.arange(HORIZON)
        ax.plot(steps, total       / 1000, label="total forecast",  lw=2.5, color="#2c3e50")
        ax.plot(steps, (level + trend) / 1000, label="level + trend", lw=1.8,
                ls="--", color="#e67e22")
        ax.plot(steps, seasonality / 1000, label="seasonality",    lw=1.5,
                ls=":",  color="#8e44ad")
        ax.set_title(f"Trend + Seasonality decomposition -- series {uid}", fontsize=11)
        ax.set_xlabel(f"weeks ahead (horizon = {HORIZON})")
        ax.set_ylabel("Weekly Sales ($000s)")
        ax.legend(fontsize=9)
    except Exception as exc:
        ax.set_title(f"Decomposition unavailable for {uid}: {exc}")
        ax.axis("off")

plt.tight_layout()
plt.savefig("/tmp/plot_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()

## Robustness check — multi-window cross-validation on the best config

In [ ]:
N_CV_WINDOWS = 4

best_row = results_df.loc[results_df["val_wmae"].idxmin()]
best_stack_types, best_n_blocks, best_mlp_units = build_architecture(
    best_row["architecture"], best_row["n_blocks_per_stack"], best_row["hidden_width"]
)

cv_model = NBEATSx(
    h=HORIZON,
    input_size=int(best_row["input_size"]),
    futr_exog_list=FUTR_EXOG,
    hist_exog_list=HIST_EXOG,
    stack_types=best_stack_types,
    n_blocks=best_n_blocks,
    mlp_units=best_mlp_units,
    dropout_prob_theta=float(best_row["dropout_prob_theta"]),
    loss=MAE(),
    max_steps=int(best_row["max_steps"]),
    learning_rate=float(best_row["learning_rate"]),
    batch_size=int(best_row["batch_size"]),
    windows_batch_size=int(best_row["windows_batch_size"]),
    scaler_type="robust",
    val_check_steps=VAL_CHECK_STEPS,
    early_stop_patience_steps=EARLY_STOP_PATIENCE_STEPS,
    start_padding_enabled=True,
    random_seed=42,
    enable_progress_bar=False,
    logger=False,
    devices=1,
    accelerator=ACCELERATOR,
    gradient_clip_val=1.0,
)

nf_cv = NeuralForecast(models=[cv_model], freq=FREQ)
nf_full_df = df[df["unique_id"].isin(valid_ids)][
    ["unique_id", "ds", "y"] + FUTR_EXOG + HIST_EXOG
].reset_index(drop=True)

cv_df = nf_cv.cross_validation(df=nf_full_df, n_windows=N_CV_WINDOWS, step_size=HORIZON)
cv_df = cv_df.reset_index() if "unique_id" not in cv_df.columns else cv_df

cv_truth = df[["unique_id", "ds", "y_raw", "sample_weight"]].rename(
    columns={"sample_weight": "sw_eval"}
)
cv_merged = cv_df.merge(cv_truth, on=["unique_id", "ds"], how="inner")
cv_merged["pred_raw"] = cv_merged["NBEATSx"].values

bad = ~np.isfinite(cv_merged["pred_raw"].values)
if bad.any():
    print(f"Warning: {bad.sum()} non-finite CV predictions; replacing with series median.")
    cv_merged = cv_merged.merge(_series_median, on="unique_id", how="left")
    cv_merged.loc[bad, "pred_raw"] = cv_merged.loc[bad, "y_median"].values
    cv_merged = cv_merged.drop(columns=["y_median"], errors="ignore")

cv_wmae_by_cutoff = (
    cv_merged.groupby("cutoff")
    .apply(lambda g: wmae(g["y_raw"], g["pred_raw"], g["sw_eval"]))
)

print("Per-window WMAE across the rolling backtest (best config):")
print(cv_wmae_by_cutoff)
print(f"\ncv_mean={cv_wmae_by_cutoff.mean():.2f}  cv_std={cv_wmae_by_cutoff.std():.2f}  "
      f"(single-holdout val_wmae was {best_row['val_wmae']:.2f})")

# ── Plot: per-window WMAE bar chart ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(d)[:10] for d in cv_wmae_by_cutoff.index], cv_wmae_by_cutoff.values,
       color="#3498db", edgecolor="white")
ax.axhline(cv_wmae_by_cutoff.mean(), color="red", ls="--", lw=1.5,
           label=f"Mean WMAE ${cv_wmae_by_cutoff.mean():,.0f}")
ax.axhline(best_row["val_wmae"], color="#2ecc71", ls="--", lw=1.5,
           label=f"Single-holdout WMAE ${best_row['val_wmae']:,.0f}")
ax.set_xlabel("CV window cutoff date")
ax.set_ylabel("WMAE ($)")
ax.set_title(f"Cross-validation robustness check ({N_CV_WINDOWS} rolling windows)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.1f}k"))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("/tmp/plot_cv_windows.png", dpi=150)
plt.show()

## Log summary run to Weights & Biases

In [ ]:
summary_run = wandb.init(project=WANDB_PROJECT, name="summary", group="summary",
                          config={"n_search_iter": N_SEARCH_ITER})
summary_run.tags = ["best_run:" + BEST_RUN_NAME, "search_type:randomized"]

wandb.log({
    "best_val_wmae":          float(best_val_wmae),
    "best_overfit_score":     float(results_df.loc[results_df["val_wmae"].idxmin(), "overfit_score"]),
    "best_overfit_level":     results_df.loc[results_df["val_wmae"].idxmin(), "overfit_level"],
    "cv_wmae_mean":           float(cv_wmae_by_cutoff.mean()),
    "cv_wmae_std":            float(cv_wmae_by_cutoff.std()),
    "ensemble_val_wmae":      float(ensemble_wmae),
    "n_severe_overfit_runs":  int((results_df["overfit_level"] == "severe").sum()),
    "n_clean_runs":           int((results_df["overfit_level"] == "clean").sum()),
})

results_csv_path = "/tmp/nbeats_training_results.csv"
results_df.to_csv(results_csv_path, index=False)
wandb.log({"results_table": wandb.Table(dataframe=results_df)})

for plot_name, plot_path in [
    ("overfitting_dashboard",     "/tmp/plot_overfit_dashboard.png"),
    ("wmae_comparison",           "/tmp/plot_wmae_comparison.png"),
    ("learning_curves",           "/tmp/plot_learning_curves.png"),
    ("actual_vs_pred",            "/tmp/plot_actual_vs_pred.png"),
    ("holiday_breakdown",         "/tmp/plot_holiday_breakdown.png"),
    ("top_series_forecast",       "/tmp/plot_top_series_forecast.png"),
    ("per_series_error",          "/tmp/plot_per_series_error.png"),
    ("hyperparam_sensitivity",    "/tmp/plot_hyperparam_sensitivity.png"),
    ("decomposition",             "/tmp/plot_decomposition.png"),
    ("cv_windows",                "/tmp/plot_cv_windows.png"),
]:
    if os.path.exists(plot_path):
        wandb.log({f"plot_{plot_name}": wandb.Image(plot_path)})

artifact = wandb.Artifact("nbeats_results", type="results")
artifact.add_file(results_csv_path)
wandb.log_artifact(artifact)

wandb.finish()
print(f"Summary run logged under project '{WANDB_PROJECT}'.")